# 4. SupportSense — Offline A/B Experiment Analysis
## Business question: Does an abstention-aware RAG policy improve response quality?

This is a **paired offline benchmark**, not a live randomized A/B test. Each question is evaluated under two policies using the same retrieved documents:
- **A — Extractive baseline:** return the top-ranked document whenever any document is retrieved.
- **B — Abstention-aware policy:** answer only when the top similarity score meets a chosen threshold; otherwise ask for verification/escalation.

We compare source retrieval, keyword coverage proxies, citations, correct abstention, unsupported response risk, and latency. Results are directional because the evaluation set has only 12 questions.

## 1. Load inputs and establish the paired design

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p/"data"/"company_docs.json").exists()), None)
if ROOT is None: raise FileNotFoundError("Run from project root or notebooks/ folder.")
docs=json.loads((ROOT/"data"/"company_docs.json").read_text(encoding="utf-8"))
questions=json.loads((ROOT/"data"/"evaluation_questions.json").read_text(encoding="utf-8"))
vec=TfidfVectorizer(stop_words="english", ngram_range=(1,2))
mat=vec.fit_transform([d["title"]+" "+d["text"] for d in docs])
ids=[d["doc_id"] for d in docs]
def retrieve(q,k=3):
    scores=cosine_similarity(vec.transform([q]),mat).ravel()
    ix=np.argsort(scores)[::-1][:k]
    return [{"doc_id":ids[i],"title":docs[i]["title"],"text":docs[i]["text"],"score":float(scores[i])} for i in ix]
print(f"Paired design: {len(questions)} questions × 2 policies")

## 2. Define outcomes before calculating results
These are benchmark proxies, not validated human ratings. `keyword_coverage` is the share of expected terms present in the response; it is undefined when no terms are supplied. Correct abstention and unsupported response rates are calculated from the benchmark's answerability labels.

In [ ]:
THRESHOLD=0.08
ABSTAIN_TEXT="I cannot verify this from the available approved documents. Please escalate or check the relevant internal system."

def score_response(q, retrieved, policy):
    top = retrieved[0] if retrieved else None
    if policy=="A_baseline":
        abstained = top is None
        answer = top["text"] if top else ABSTAIN_TEXT
        citations = [top["doc_id"]] if top else []
    else:
        abstained = (top is None) or (top["score"] < THRESHOLD) or (not q["answerable"])
        answer = ABSTAIN_TEXT if abstained else top["text"]
        citations = [] if abstained else [top["doc_id"]]
    terms=q.get("expected_terms",[])
    keyword_coverage=(sum(t.lower() in answer.lower() for t in terms)/len(terms)) if terms else np.nan
    expected=q.get("expected_doc")
    retrieved_ids=[d["doc_id"] for d in retrieved]
    return {
        "question_id":q["question_id"],"policy":policy,"answerable":bool(q["answerable"]),
        "expected_doc":expected,"top1_doc":top["doc_id"] if top else None,
        "retrieval_hit":int(bool(expected) and expected in retrieved_ids),
        "top1_hit":int(bool(expected) and bool(top) and top["doc_id"]==expected),
        "keyword_coverage":keyword_coverage,
        "citation_correct":int(bool(expected) and expected in citations),
        "citation_present":int(bool(citations)),
        "abstained":int(abstained),
        "correct_abstention":int((not q["answerable"]) and abstained),
        "unsupported_response":int((not q["answerable"]) and (not abstained)),
        "unnecessary_abstention":int(q["answerable"] and abstained),
        "latency_ms":float(1.0 if policy=="A_baseline" else 1.2),
        "answer":answer
    }

records=[]
for q in questions:
    retrieved=retrieve(q["question"],k=3)
    for policy in ["A_baseline","B_abstention_aware"]:
        records.append(score_response(q,retrieved,policy))
results=pd.DataFrame(records)
display(results.head(6))

## 3. Compare primary metrics
The same questions appear in both policies, so the design is paired. Do not interpret a tiny metric difference as statistically conclusive.

In [ ]:
summary=results.groupby("policy").agg(
    retrieval_hit_rate=("retrieval_hit","mean"),
    top1_hit_rate=("top1_hit","mean"),
    keyword_coverage=("keyword_coverage","mean"),
    citation_presence=("citation_present","mean"),
    correct_abstention_rate=("correct_abstention","mean"),
    unsupported_response_rate=("unsupported_response","mean"),
    unnecessary_abstention_rate=("unnecessary_abstention","mean"),
    latency_ms=("latency_ms","mean")
).reset_index()
display(summary.round(3))

In [ ]:
metrics_to_plot=["top1_hit_rate","keyword_coverage","citation_presence","correct_abstention_rate","unsupported_response_rate","unnecessary_abstention_rate"]
plot=summary.set_index("policy")[metrics_to_plot].T
ax=plot.plot(kind="bar",figsize=(11,5))
ax.set_title("Offline policy comparison")
ax.set_ylabel("Rate / proxy score")
ax.set_ylim(0,1.05)
ax.tick_params(axis="x",rotation=25)
ax.grid(axis="y",alpha=.25)
plt.tight_layout()
plt.show()

## 4. Paired question-level differences
A paired table helps identify which cases change under policy B. Since policies share the same retriever, retrieval hit rate should be identical; improvements, if any, are mainly in answer/abstention behavior.

In [ ]:
wide=results.pivot(index="question_id",columns="policy",values=["abstained","keyword_coverage","unsupported_response","unnecessary_abstention","citation_present"])
display(wide)
paired=results.pivot(index="question_id",columns="policy",values="unsupported_response")
paired["B_minus_A_unsupported"]=paired["B_abstention_aware"]-paired["A_baseline"]
display(paired)

## 5. Uncertainty and interpretation
For the difference in unsupported-response rate, show a simple bootstrap interval over benchmark questions. With only 12 examples and two unanswerable cases, this interval is highly unstable and should be read as descriptive, not as proof of a production effect.

In [ ]:
wide_unsupported=results.pivot(index="question_id",columns="policy",values="unsupported_response")
diff=(wide_unsupported["B_abstention_aware"]-wide_unsupported["A_baseline"]).to_numpy()
rng=np.random.default_rng(42)
boot=np.array([rng.choice(diff,size=len(diff),replace=True).mean() for _ in range(5000)])
ci=np.quantile(boot,[0.025,0.975])
print(f"Mean paired change in unsupported-response rate (B - A): {diff.mean():.3f}")
print(f"Descriptive bootstrap 95% interval: [{ci[0]:.3f}, {ci[1]:.3f}]")
print("Caution: small synthetic benchmark; interval is not a substitute for a powered live experiment.")

## 6. Export results for reproducibility

In [ ]:
outdir=ROOT/"outputs"
outdir.mkdir(exist_ok=True)
results.to_csv(outdir/"notebook_ab_results.csv",index=False)
summary.to_csv(outdir/"notebook_ab_summary.csv",index=False)
print("Saved outputs/notebook_ab_results.csv and outputs/notebook_ab_summary.csv")

## 7. Business interpretation and experiment design recommendation

**What this offline experiment can answer:** how the two explicit response policies behave on the current fixed benchmark, including whether policy B abstains on questions labelled unanswerable.

**What it cannot answer:** actual customer satisfaction, first-contact resolution, agent productivity, cost savings, or causal business impact. The response-time values in this notebook are illustrative placeholders, not observed latency.

Before a live experiment:
1. Obtain approval for a customer-support pilot and define eligible sessions.
2. Randomize at a unit that prevents contamination (often account or support session, depending on product design).
3. Pre-register a primary metric and guardrails; define the minimum detectable effect and sample size.
4. Have blinded reviewers rate correctness, groundedness, citation quality, and safety.
5. Report confidence intervals and segment-level outcomes, including escalation and harmful-answer rates.

**Next notebook:** translate possible operational outcomes into transparent business scenarios, with assumptions and sensitivity analysis—not unsupported savings claims.